# Đăng video lên GayVideo.fun (web + X) từ Google Photos

Notebook này lấy video từ **link chia sẻ Google Photos** `photos.app.goo.gl/…` (một video hoặc một album) rồi xử lý từng video. Link Google Drive chỉ là **phương án phụ**, dùng khi Google Photos chặn.

1. tải **file gốc**:
   - theo redirect của link rút gọn tới trang chia sẻ;
   - đọc các URL `lh3.googleusercontent.com` của video;
   - tải bản gốc bằng hậu tố `=dv`;
   - kiểm tra bằng **ffprobe** rằng file là video thật, không phải ảnh xem trước.
2. gắn watermark GayVideo.fun (giống luồng migration hiện tại);
3. upload lên **host chính**, và các host phụ nếu bật, để lấy mã file;
4. tạo thumbnail;
5. gọi **publish API**: lên web, và nếu bật thì vào hàng đợi đăng **X @Gayvideofun** qua Buffer, với clip ngắn cắt bằng ffmpeg và caption tiếng Anh.

### Chuẩn bị (một lần)
- **Colab Secrets** (biểu tượng 🔑 bên trái, bật *Notebook access* cho từng secret):
  - `GAYVIDEO_API_TOKEN`: token tạo trên VPS (`npm run api-token -- create …`);
  - `HOST_DOOD_API_KEY`, `HOST_VOE_API_KEY`, `HOST_EARNVIDS_API_KEY`: key của các host. Bắt buộc có key của host chính; key host phụ chỉ cần khi bật `UPLOAD_MIRRORS`.
- **Link Google Photos** phải bật chia sẻ: *Chia sẻ → Tạo đường liên kết*.

### Cách dùng
1. Chạy ô **Cài đặt**.
2. Chỉnh ô **Cấu hình**.
3. Dán link vào ô **Danh sách video**, mỗi dòng một link, hoặc trỏ tới file CSV.
4. **Đánh dấu ô xác nhận.** Không xác nhận thì notebook không chạy.
5. Chạy ô **Đăng**. Cuối cùng có bảng kết quả cho từng video.

### Cam kết của notebook
- **Không đăng trùng:** mỗi video đã đăng được ghi vào sổ `publish_log.csv` trên Google Drive (theo link và theo sha256 của file). Chạy lại thì các video đó được bỏ qua.
- **Lỗi một video không dừng cả lô:** lỗi được ghi vào bảng, rồi notebook làm tiếp video sau.
- **Dọn file tạm:** file tạm của mỗi video bị xoá ngay sau khi xong video đó.
- **Không lộ key:** key được đọc từ Colab Secrets và **không in ra**; thông báo lỗi luôn che key.

### Nếu Google Photos chặn
Google Photos không có API công khai để tải video từ link chia sẻ, và yt-dlp cũng không hỗ trợ (chỉ hỗ trợ Google Drive). Notebook dùng bộ đọc riêng trong `colab_helpers.py`:
- mỗi bước tự thử lại 3 lần;
- báo lỗi rõ cho từng link: chưa chia sẻ hoặc bị chặn đăng nhập, link đã xoá, album không có video, hoặc trang đã đổi giao diện;
- khi Google đổi giao diện, sửa `photos_media_urls()` và HTML mẫu trong `testdata/`, rồi chạy test (`python3 -m unittest discover -s scripts/publish-client`).

Nếu một link vẫn báo lỗi tải:
1. tải video đó lên **Google Drive**;
2. chia sẻ *Bất kỳ ai có đường liên kết*;
3. dán link Drive vào danh sách thay cho link Photos.

### Thư viện (tự sinh: hai ô dưới ghi file client và helpers vào /content)

In [ ]:
%%writefile /content/gayvideo_publish.py
# Client publish API: tự sinh từ repo (build_notebook.py), đừng sửa ở đây.
"""Client for the GayVideo.fun publish API (docs/API.md).

Standard library only, so it runs as-is on Google Colab and on a laptop
(Python 3.8+). Two ways to use it:

    from gayvideo_publish import GayVideoClient
    client = GayVideoClient(token="gv_...")          # or env GAYVIDEO_API_TOKEN
    result = client.publish(
        title="Bevis Trần Đức Vương – My love story",
        mirrors={"dood": "abc123", "voe": "xyz789"},
        tags=["Bevis", "Couple"],
        category="couple",
        thumbnail="cover.jpg",                        # local file or https URL
        publish=False,                                # True = public right away
    )
    print(result["video"]["url"])

Also announce it on X (@Gayvideofun, through Buffer): pass an English
caption and the source video; a short clip (X_CLIP_SECONDS, from the
server) is cut here with ffmpeg, uploaded, and posted once the video is
public, with a link to its page:

    client.publish(..., publish=True,
                   x_caption="New: Bevis & Pinky's love story 🔥",
                   x_video="/content/full.mp4", x_clip_start=30)

    python gayvideo_publish.py --title "..." --mirror dood:abc123 --tag Bevis --publish \
        --x-caption "New video" --x-video full.mp4
"""

from __future__ import annotations

import argparse
import base64
import json
import mimetypes
import os
import shutil
import subprocess
import sys
import tempfile
import urllib.error
import urllib.request
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple, Union

DEFAULT_BASE_URL = "https://gayvideo.fun"
MAX_THUMBNAIL_BYTES = 8 * 1024 * 1024

Mirrors = Union[Mapping[str, str], Iterable[Tuple[str, str]]]


class PublishError(Exception):
    """The API refused the request. `status`, `code` and `body` say why."""

    def __init__(self, status: int, code: str, message: str, body: Optional[dict] = None):
        super().__init__(f"{status} {code}: {message}")
        self.status = status
        self.code = code
        self.body = body or {}


class GayVideoClient:
    def __init__(
        self,
        token: Optional[str] = None,
        base_url: Optional[str] = None,
        timeout: float = 120,
    ) -> None:
        self.token = (token or os.environ.get("GAYVIDEO_API_TOKEN") or "").strip()
        if not self.token.startswith("gv_"):
            raise ValueError("Missing token: pass token='gv_...' or set GAYVIDEO_API_TOKEN")
        self.base_url = (base_url or os.environ.get("GAYVIDEO_API_URL") or DEFAULT_BASE_URL).rstrip("/")
        self.timeout = timeout

    def _request(
        self,
        method: str,
        path: str,
        body: Optional[dict] = None,
        *,
        upload: Optional[str] = None,
    ) -> dict:
        headers = {
            "Authorization": f"Bearer {self.token}",
            "Accept": "application/json",
            "User-Agent": "gayvideo-publish/2",
        }
        data: Any = None
        handle = None
        if upload is not None:
            # Streamed from disk, not read into memory.
            handle = open(upload, "rb")
            data = handle
            headers["Content-Type"] = "video/mp4"
            headers["Content-Length"] = str(os.path.getsize(upload))
        elif body is not None:
            data = json.dumps(body).encode("utf-8")
            headers["Content-Type"] = "application/json"
        request = urllib.request.Request(self.base_url + path, data=data, method=method, headers=headers)
        try:
            with urllib.request.urlopen(request, timeout=self.timeout) as response:
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as error:
            raw = error.read().decode("utf-8", "replace")
            try:
                payload = json.loads(raw)
            except ValueError:
                payload = {}
            info = payload.get("error", {}) if isinstance(payload, dict) else {}
            raise PublishError(
                error.code,
                info.get("code", "http_error"),
                info.get("message", raw[:200] or error.reason),
                payload if isinstance(payload, dict) else None,
            ) from None
        finally:
            if handle is not None:
                handle.close()

    def meta(self) -> dict:
        """Enabled hosts and category slugs; also checks the token works."""
        return self._request("GET", "/api/v1/meta")

    def get_video(self, video_id: str) -> dict:
        return self._request("GET", f"/api/v1/videos/{video_id}")["video"]

    def upload_clip(self, clip_path: str) -> dict:
        """Upload an MP4 made by cut_clip(); returns {"clipId", "bytes", "seconds"}."""
        return self._request("POST", "/api/v1/clips", upload=clip_path)

    def clip_seconds(self) -> int:
        """X_CLIP_SECONDS: from the environment if set, else the server's (0 = full video)."""
        local = os.environ.get("X_CLIP_SECONDS", "").strip()
        if local:
            return int(local)
        return int(self.meta().get("x", {}).get("clipSeconds", 120))

    def publish(
        self,
        title: str,
        mirrors: Mirrors,
        *,
        slug: Optional[str] = None,
        description: Optional[str] = None,
        category: Optional[str] = None,
        tags: Optional[Sequence[str]] = None,
        quality: Optional[str] = None,
        thumbnail: Optional[str] = None,
        publish: bool = False,
        x_caption: Optional[str] = None,
        x_video: Optional[str] = None,
        x_clip: Optional[str] = None,
        x_clip_start: float = 0,
        x_clip_seconds: Optional[int] = None,
    ) -> dict:
        """Create the video, or update the draft its files already belong to.

        For X, pass x_caption (English) and either x_video (the source file;
        a clip is cut from x_clip_start) or x_clip (an MP4 you cut yourself).
        The X post goes out once the video is public; it never blocks the web.

        Returns {"created": bool, "video": {...}, "x": {...}?, "warnings": [...]}.
        Raises PublishError on refusal (e.g. 409 already_published).
        """
        clip_id = None
        if x_caption:
            if not (x_video or x_clip):
                raise ValueError("x_caption needs x_video (source to cut) or x_clip (ready MP4)")
            if x_clip:
                clip_id = self.upload_clip(x_clip)["clipId"]
            else:
                seconds = self.clip_seconds() if x_clip_seconds is None else x_clip_seconds
                workdir = tempfile.mkdtemp(prefix="gv-clip-")
                try:
                    clip_path = cut_clip(x_video, os.path.join(workdir, "clip.mp4"),
                                         start=x_clip_start, seconds=seconds)
                    clip_id = self.upload_clip(clip_path)["clipId"]
                finally:
                    shutil.rmtree(workdir, ignore_errors=True)

        pairs = list(mirrors.items()) if isinstance(mirrors, Mapping) else list(mirrors)
        body: Dict[str, Any] = {
            "title": title,
            "mirrors": [{"host": host, "fileCode": code} for host, code in pairs],
            "publish": publish,
        }
        if slug:
            body["slug"] = slug
        if description is not None:
            body["description"] = description
        if category:
            body["categorySlug"] = category
        if tags is not None:
            body["tags"] = list(tags)
        if quality:
            body["quality"] = quality
        if thumbnail:
            if thumbnail.startswith(("http://", "https://")):
                body["thumbnailUrl"] = thumbnail
            else:
                body["thumbnailDataUrl"] = image_data_url(thumbnail)
        if clip_id:
            body["x"] = {"caption": x_caption, "clipId": clip_id}
        return self._request("POST", "/api/v1/videos", body)


def cut_clip(source: str, output: str, *, start: float = 0, seconds: int = 120) -> str:
    """Cut and re-encode a clip X accepts: MP4, H.264 High (yuv420p), AAC-LC
    stereo 44.1 kHz, longest side at most 1280 and 60 fps, moov at the start
    (+faststart). seconds=0 keeps the whole video. Needs ffmpeg on PATH
    (Colab has it; Windows: winget install Gyan.FFmpeg; macOS: brew install ffmpeg).
    """
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg is not installed (see docs/API.md)")
    command = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y"]
    if start:
        command += ["-ss", str(start)]
    command += ["-i", source]
    if seconds:
        command += ["-t", str(seconds)]
    command += [
        "-map", "0:v:0", "-map", "0:a:0?",
        # Longest side at most 1280 (720p landscape or portrait), even sizes.
        "-vf", "scale='if(gte(iw,ih),min(1280,iw),-2)':'if(gte(iw,ih),-2,min(1280,ih))'",
        "-fpsmax", "60",
        "-c:v", "libx264", "-profile:v", "high", "-level:v", "4.1", "-pix_fmt", "yuv420p",
        "-preset", "medium", "-crf", "23", "-maxrate", "6M", "-bufsize", "12M",
        "-c:a", "aac", "-b:a", "128k", "-ar", "44100", "-ac", "2",
        "-movflags", "+faststart",
        output,
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(output):
        raise RuntimeError(f"ffmpeg failed: {result.stderr.strip()[-500:]}")
    return output


def image_data_url(path: str) -> str:
    size = os.path.getsize(path)
    if size > MAX_THUMBNAIL_BYTES:
        raise ValueError(f"{path} is {size} bytes; thumbnails are limited to 8 MB")
    mime = mimetypes.guess_type(path)[0] or ""
    if mime not in ("image/jpeg", "image/png", "image/webp", "image/gif"):
        raise ValueError(f"{path}: use a .jpg, .png, .webp or .gif image")
    with open(path, "rb") as handle:
        return f"data:{mime};base64," + base64.b64encode(handle.read()).decode("ascii")


def _parse_mirror(value: str) -> Tuple[str, str]:
    host, sep, code = value.partition(":")
    if not sep or not host or not code:
        raise argparse.ArgumentTypeError("use host:fileCode, e.g. dood:abc123")
    return host.strip(), code.strip()


def main(argv: Optional[List[str]] = None) -> int:
    parser = argparse.ArgumentParser(description="Publish a video to GayVideo.fun")
    parser.add_argument("--title", required=True)
    parser.add_argument("--mirror", action="append", type=_parse_mirror, required=True,
                        help="host:fileCode, repeat per host (dood, voe, earnvids)")
    parser.add_argument("--slug")
    parser.add_argument("--description")
    parser.add_argument("--category", help="category slug (see --meta)")
    parser.add_argument("--tag", action="append", dest="tags")
    parser.add_argument("--quality", choices=["HD", "4K"])
    parser.add_argument("--thumbnail", help="local image file or https URL")
    parser.add_argument("--publish", action="store_true", help="make it public now (default: draft)")
    parser.add_argument("--x-caption", help="also post on X with this English caption")
    parser.add_argument("--x-video", help="source video to cut the X clip from")
    parser.add_argument("--x-clip", help="a ready MP4 clip (skips cutting)")
    parser.add_argument("--x-start", type=float, default=0, help="clip start, in seconds")
    parser.add_argument("--x-seconds", type=int, default=None,
                        help="clip length (default: X_CLIP_SECONDS from the server; 0 = full)")
    parser.add_argument("--base-url", default=None)
    args = parser.parse_args(argv)

    client = GayVideoClient(base_url=args.base_url)
    try:
        result = client.publish(
            args.title,
            args.mirror,
            slug=args.slug,
            description=args.description,
            category=args.category,
            tags=args.tags,
            quality=args.quality,
            thumbnail=args.thumbnail,
            publish=args.publish,
            x_caption=args.x_caption,
            x_video=args.x_video,
            x_clip=args.x_clip,
            x_clip_start=args.x_start,
            x_clip_seconds=args.x_seconds,
        )
    except PublishError as error:
        print(f"Refused: {error}", file=sys.stderr)
        return 1
    video = result["video"]
    state = "published" if video["published"] else "draft (approve it in /admin/review)"
    print(f"{'Created' if result['created'] else 'Updated'}: {client.base_url}{video['url']} [{state}]")
    if "x" in result:
        print(f"X: {result['x']['status']} (posted through Buffer once public)")
    for warning in result.get("warnings", []):
        print(f"warning: {warning}", file=sys.stderr)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/colab_helpers.py
# Helpers tải/upload: tự sinh từ repo (build_notebook.py), đừng sửa ở đây.
"""Helpers for colab_publish.ipynb: get a source video, upload it to the
hosts, and keep a ledger so nothing is posted twice.

Kept separate from the notebook so it can be tested (test_colab_helpers.py);
build_notebook.py embeds this file into the notebook. Needs requests (on
Colab by default), curl, ffmpeg/ffprobe, and for watermarks rsvg-convert
(apt librsvg2-bin) and Pillow.

Secrets (host API keys) are never printed: they are passed to curl in a
config file readable only by this user, and redact() strips them from every
error message.
"""

from __future__ import annotations

import csv
import datetime as dt
import hashlib
import html
import json
import os
import re
import shutil
import subprocess
import tempfile
import time
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence
from urllib.parse import urlparse

import requests

BROWSER_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"
)
_SECRETS: List[str] = []


# --- secrets -----------------------------------------------------------------

def register_secret(value: Optional[str]) -> Optional[str]:
    """Remember a secret so redact() hides it; returns it unchanged."""
    if value and len(value) >= 4 and value not in _SECRETS:
        _SECRETS.append(value)
    return value


def redact(text: object, limit: int = 600) -> str:
    out = str(text)
    for secret in _SECRETS:
        out = out.replace(secret, "***")
    out = re.sub(r"(?i)(key|api_key|token)=([^&\s\"']+)", r"\1=***", out)
    return out.replace("\r", " ").replace("\n", " ").strip()[:limit]


# --- input --------------------------------------------------------------------

@dataclass
class Job:
    """One line of input: a share link plus optional metadata."""

    link: str
    title: str = ""
    tags: List[str] = field(default_factory=list)
    category: str = ""
    x_caption: str = ""
    clip_start: Optional[float] = None


def _split_tags(value: str) -> List[str]:
    return [tag.strip() for tag in re.split(r"[;,]", value or "") if tag.strip()]


def parse_lines(text: str) -> List[Job]:
    """`link | title | tags | category | x caption | clip start`, all but the
    link optional; blank lines and lines starting with # are ignored."""
    jobs = []
    for raw in (text or "").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        parts = [part.strip() for part in line.split("|")]
        parts += [""] * (6 - len(parts))
        start = parts[5]
        jobs.append(
            Job(
                link=parts[0],
                title=parts[1],
                tags=_split_tags(parts[2]),
                category=parts[3],
                x_caption=parts[4],
                clip_start=float(start) if start else None,
            )
        )
    return jobs


def parse_csv(path: str) -> List[Job]:
    """CSV with a header: link (required), title, tags, category, x_caption, clip_start."""
    jobs = []
    with open(path, newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            link = (row.get("link") or "").strip()
            if not link:
                continue
            start = (row.get("clip_start") or "").strip()
            jobs.append(
                Job(
                    link=link,
                    title=(row.get("title") or "").strip(),
                    tags=_split_tags(row.get("tags") or ""),
                    category=(row.get("category") or "").strip(),
                    x_caption=(row.get("x_caption") or "").strip(),
                    clip_start=float(start) if start else None,
                )
            )
    return jobs


def ascii_text(value: str) -> str:
    """Accents stripped, đ -> d: for English captions built from Vietnamese titles."""
    folded = unicodedata.normalize("NFD", value)
    folded = "".join(ch for ch in folded if unicodedata.category(ch) != "Mn")
    return folded.replace("đ", "d").replace("Đ", "D")


# --- sources: Google Photos and Google Drive -------------------------------------

class SourceError(RuntimeError):
    """The link could not be read; the message says what to do instead."""


@dataclass
class SourceItem:
    """One video found behind a link. `key` identifies it for the ledger."""

    key: str
    download: Callable[[Path], Path]
    label: str = ""


_PHOTOS_MEDIA = re.compile(r"https://lh3\.googleusercontent\.com/pw/[A-Za-z0-9_-]{20,}")
_DRIVE_FILE = re.compile(r"/file/d/([A-Za-z0-9_-]{10,})|[?&]id=([A-Za-z0-9_-]{10,})")
_DRIVE_FOLDER = re.compile(r"/folders/([A-Za-z0-9_-]{10,})")

PHOTOS_HELP = (
    "Hãy kiểm tra link Google Photos đã bật chia sẻ công khai (ai có link đều xem được). "
    "Nếu vẫn lỗi, Google Photos có thể đang chặn tải tự động: hãy tải video lên Google Drive, "
    "chia sẻ 'Bất kỳ ai có đường liên kết', rồi dùng link Drive thay thế."
)


def link_kind(link: str) -> str:
    host = urlparse(link).netloc.lower()
    if host in ("photos.app.goo.gl", "photos.google.com", "goo.gl"):
        return "photos"
    if host in ("drive.google.com", "docs.google.com"):
        return "drive"
    return "unknown"


def photos_media_urls(page: str) -> List[str]:
    """Distinct lh3 media base URLs on a shared Google Photos page (video,
    photo or album), in page order, without size suffixes."""
    seen, out = set(), []
    for url in _PHOTOS_MEDIA.findall(html.unescape(page).replace("\\u003d", "=").replace("\\/", "/")):
        if url not in seen:
            seen.add(url)
            out.append(url)
    return out


def photos_album_title(page: str) -> str:
    match = re.search(r'<meta property="og:title" content="([^"]*)"', page)
    title = html.unescape(match.group(1)).strip() if match else ""
    return "" if title.lower() in ("", "google photos", "photo", "video") else title


def _retry(action: Callable[[], Any], what: str, attempts: int = 3, base_delay: float = 3.0) -> Any:
    """Runs `action`, retrying network errors and _Retryable (429/5xx) with backoff."""
    last: Optional[BaseException] = None
    for attempt in range(attempts):
        try:
            return action()
        except (requests.RequestException, _Retryable) as error:
            last = error
            if attempt < attempts - 1:
                time.sleep(base_delay * (2 ** attempt))
    raise SourceError(f"{what} thất bại sau {attempts} lần thử ({redact(last)}). {PHOTOS_HELP}")


class _Retryable(Exception):
    pass


def _photos_page(link: str, session: requests.Session) -> str:
    """Follows the short link (photos.app.goo.gl -> photos.google.com/share/…)
    and returns the share page, or explains why it can't."""

    def fetch() -> requests.Response:
        response = session.get(link, headers={"User-Agent": BROWSER_UA}, timeout=60, allow_redirects=True)
        if response.status_code == 429 or response.status_code >= 500:
            raise _Retryable(f"HTTP {response.status_code}")
        return response

    response = _retry(fetch, "Mở link Google Photos")
    final = urlparse(response.url)
    if final.netloc.startswith(("accounts.google.com", "consent.google.com")) or response.status_code in (401, 403):
        raise SourceError(f"Google Photos yêu cầu đăng nhập/đồng ý cookie (link chưa chia sẻ công khai hoặc bị chặn). {PHOTOS_HELP}")
    if response.status_code == 404:
        raise SourceError(f"Link Google Photos không tồn tại hoặc đã bị xoá (HTTP 404). {PHOTOS_HELP}")
    if response.status_code >= 400:
        raise SourceError(f"Google Photos trả HTTP {response.status_code}. {PHOTOS_HELP}")
    if final.netloc != "photos.google.com" or not final.path.startswith("/share/"):
        raise SourceError(f"Link không dẫn tới trang chia sẻ Google Photos (tới {final.netloc}{final.path[:40]}). {PHOTOS_HELP}")
    return response.text


IMAGE_CODECS = {"mjpeg", "png", "webp", "gif", "bmp", "tiff", "heif", "hevc_image", "av1_image"}


def verify_video(path: Path, min_seconds: float = 1.0) -> Dict[str, Any]:
    """ffprobe the download: it must hold a real video stream (not a still
    preview image) at least `min_seconds` long."""
    try:
        out = run(["ffprobe", "-v", "error", "-show_entries",
                   "stream=codec_type,codec_name,width,height:format=duration,format_name",
                   "-of", "json", str(path)], timeout=300)
        payload = json.loads(out)
    except Exception as error:
        raise SourceError(f"File tải về không đọc được bằng ffprobe ({redact(error, 200)}). {PHOTOS_HELP}") from None
    videos = [s for s in payload.get("streams", []) if s.get("codec_type") == "video"]
    duration = float(payload.get("format", {}).get("duration") or 0)
    if not videos or videos[0].get("codec_name") in IMAGE_CODECS or duration < min_seconds:
        kind = videos[0].get("codec_name") if videos else "không có luồng video"
        raise SourceError(f"File tải về là ảnh xem trước chứ không phải video ({kind}, {duration:.1f}s). {PHOTOS_HELP}")
    return {"duration": duration, "codec": videos[0].get("codec_name"),
            "width": videos[0].get("width"), "height": videos[0].get("height")}


def _download_photos_video(base: str, session: requests.Session) -> Callable[[Path], Path]:
    def download(target: Path) -> Path:
        # "=dv" asks Google for the original video file (undocumented, stable for years).
        def fetch() -> Path:
            with session.get(base + "=dv", headers={"User-Agent": BROWSER_UA}, stream=True, timeout=(30, 600)) as response:
                if response.status_code == 429 or response.status_code >= 500:
                    raise _Retryable(f"HTTP {response.status_code}")
                kind = response.headers.get("content-type", "")
                if response.status_code >= 400 or not kind.startswith("video/"):
                    raise SourceError(
                        f"Google Photos không trả video gốc (HTTP {response.status_code}, {kind or 'không rõ loại'}). {PHOTOS_HELP}"
                    )
                with open(target, "wb") as handle:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        handle.write(chunk)
            return target

        _retry(fetch, "Tải video gốc từ Google Photos")
        verify_video(target)
        return target

    return download


def _is_photos_video(base: str, session: requests.Session) -> bool:
    """Asks for one byte of "=dv": videos answer video/*, photos do not."""

    def fetch() -> bool:
        with session.get(base + "=dv", headers={"User-Agent": BROWSER_UA, "Range": "bytes=0-0"},
                         stream=True, timeout=60) as response:
            if response.status_code == 429 or response.status_code >= 500:
                raise _Retryable(f"HTTP {response.status_code}")
            return response.status_code < 400 and response.headers.get("content-type", "").startswith("video/")

    return bool(_retry(fetch, "Kiểm tra loại mục Google Photos"))


def photos_items(link: str, session: Optional[requests.Session] = None) -> tuple:
    """(album or item title, [SourceItem]) for every video behind a Photos link."""
    session = session or requests.Session()
    page = _photos_page(link, session)
    urls = photos_media_urls(page)
    if not urls:
        raise SourceError(
            "Không tìm thấy URL ảnh/video (lh3.googleusercontent.com/pw/…) trong trang chia sẻ: "
            "có thể Google đã đổi giao diện; cập nhật photos_media_urls() và HTML mẫu trong test. " + PHOTOS_HELP
        )
    videos = [url for url in urls if _is_photos_video(url, session)]
    if not videos:
        raise SourceError(f"Link Google Photos không có video nào tải được ({len(urls)} mục, đều là ảnh hoặc bị chặn). {PHOTOS_HELP}")
    items = [
        SourceItem(
            key="photos:" + hashlib.sha256(url.split("/pw/", 1)[1].encode()).hexdigest()[:24],
            download=_download_photos_video(url, session),
            label=f"{index + 1}/{len(videos)}",
        )
        for index, url in enumerate(videos)
    ]
    return photos_album_title(page), items


def drive_items(link: str) -> tuple:
    """A Drive file or folder shared as "anyone with the link" (uses gdown)."""
    try:
        import gdown  # noqa: F401  (pip install gdown; preinstalled on Colab)
    except ImportError:
        raise SourceError("Thiếu thư viện gdown: chạy `pip install gdown`.") from None
    import gdown

    folder = _DRIVE_FOLDER.search(link)
    if folder:
        workdir = Path(tempfile.mkdtemp(prefix="gv-drive-list-"))
        try:
            files = gdown.download_folder(url=link, output=str(workdir), quiet=True, skip_download=True)
        except Exception as error:  # gdown raises many types
            shutil.rmtree(workdir, ignore_errors=True)
            raise SourceError(f"Không đọc được thư mục Drive ({redact(error)}). Kiểm tra quyền 'Bất kỳ ai có đường liên kết'.") from None
        shutil.rmtree(workdir, ignore_errors=True)
        videos = [item for item in files or [] if re.search(r"\.(mp4|mov|m4v|mkv|webm|avi)$", item.path, re.I)]
        return "", [
            SourceItem(key=f"drive:{item.id}", download=_drive_download(item.id), label=Path(item.path).name)
            for item in videos
        ]
    match = _DRIVE_FILE.search(link)
    if not match:
        raise SourceError("Link Google Drive không có mã file/thư mục.")
    file_id = match.group(1) or match.group(2)
    return "", [SourceItem(key=f"drive:{file_id}", download=_drive_download(file_id))]


def _drive_download(file_id: str) -> Callable[[Path], Path]:
    def download(target: Path) -> Path:
        import gdown

        result = gdown.download(id=file_id, output=str(target), quiet=True)
        if not result or not target.exists() or target.stat().st_size < 10_000:
            raise SourceError("Tải file từ Google Drive thất bại (link chưa chia sẻ công khai, hoặc vượt hạn mức tải của Drive).")
        verify_video(target)
        return target

    return download


def source_items(link: str) -> tuple:
    kind = link_kind(link)
    if kind == "photos":
        return photos_items(link)
    if kind == "drive":
        return drive_items(link)
    raise SourceError("Chỉ hỗ trợ link Google Photos (photos.app.goo.gl/…) hoặc Google Drive.")


# --- video tools ------------------------------------------------------------------

def run(command: Sequence[str], timeout: int = 6 * 60 * 60) -> str:
    result = subprocess.run(list(command), capture_output=True, text=True, timeout=timeout)
    if result.returncode != 0:
        raise RuntimeError(f"{Path(command[0]).name} failed: {redact(result.stderr[-800:])}")
    return result.stdout


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def probe(path: Path) -> Dict[str, Any]:
    """{"duration": seconds, "width", "height"}; raises if there is no video stream."""
    out = run(["ffprobe", "-v", "error", "-select_streams", "v:0",
               "-show_entries", "stream=width,height:format=duration", "-of", "json", str(path)], timeout=300)
    payload = json.loads(out)
    streams = payload.get("streams") or []
    if not streams:
        raise RuntimeError("Không có luồng video trong file tải về.")
    return {
        "duration": float(payload.get("format", {}).get("duration") or 0),
        "width": int(streams[0]["width"]),
        "height": int(streams[0]["height"]),
    }


def make_thumbnail(source: Path, target: Path, duration: float, at: Optional[float] = None) -> Path:
    """A JPEG frame (default: 20% into the video, at least 1 s in)."""
    second = at if at is not None else max(1.0, duration * 0.2)
    second = min(second, max(0.0, duration - 0.5))
    run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-ss", f"{second:.2f}", "-i", str(source),
         "-frames:v", "1", "-vf", "scale='min(1280,iw)':-2", "-q:v", "2", str(target)], timeout=300)
    return target


def watermark(source: Path, target: Path, logo_svg: Path, width: int) -> Path:
    """Same mark as the migration pipeline (scripts/video-migration/common.py):
    the logo with "GayVideo.fun" under it, top right, 82% opacity."""
    from PIL import Image, ImageDraw, ImageFont

    logo_size = max(48, min(150, round(width * 0.07)))
    font_size = max(12, min(28, round(width * 0.014)))
    margin = max(12, round(width * 0.0125))
    logo_png = target.with_name("logo.png")
    run(["rsvg-convert", "-w", str(logo_size), "-h", str(logo_size), "-o", str(logo_png), str(logo_svg)], timeout=120)

    text_width = round(font_size * 7.4)
    canvas = Image.new("RGBA", (max(logo_size, text_width), logo_size + round(font_size * 1.65) + 4), (0, 0, 0, 0))
    logo = Image.open(logo_png).convert("RGBA")
    canvas.alpha_composite(logo, ((canvas.width - logo.width) // 2, 0))
    font = None
    for candidate in ("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"):
        if os.path.exists(candidate):
            font = ImageFont.truetype(candidate, font_size)
            break
    draw = ImageDraw.Draw(canvas)
    text = "GayVideo.fun"
    box = draw.textbbox((0, 0), text, font=font)
    draw.text(((canvas.width - (box[2] - box[0])) // 2, canvas.height - (box[3] - box[1]) - 4), text,
              font=font, fill=(255, 255, 255, 255), stroke_width=1, stroke_fill=(0, 0, 0, 153))
    alpha = canvas.getchannel("A").point(lambda value: round(value * 0.82))
    canvas.putalpha(alpha)
    mark = target.with_name("watermark.png")
    canvas.save(mark)

    run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(source), "-i", str(mark),
         "-filter_complex", f"[1:v]format=rgba[wm];[0:v][wm]overlay=W-w-{margin}:{margin}[v]",
         "-map", "[v]", "-map", "0:a?", "-c:v", "libx264", "-preset", "veryfast", "-crf", "21",
         "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "160k", "-movflags", "+faststart", str(target)])
    return target


# --- hosts --------------------------------------------------------------------------

HOSTS = {
    "dood": {"label": "DoodStream", "api": "https://doodapi.co/api", "secret": "HOST_DOOD_API_KEY"},
    "voe": {"label": "VOE", "api": "https://voe.sx/api", "secret": "HOST_VOE_API_KEY"},
    "earnvids": {"label": "EarnVids", "api": "https://earnvidsapi.com/api", "secret": "HOST_EARNVIDS_API_KEY"},
}


def _find_value(value: Any, keys: Iterable[str]) -> Any:
    keys = set(keys)
    if isinstance(value, dict):
        for key, item in value.items():
            if key in keys and item not in (None, ""):
                return item
        for item in value.values():
            found = _find_value(item, keys)
            if found is not None:
                return found
    if isinstance(value, list):
        for item in value:
            found = _find_value(item, keys)
            if found is not None:
                return found
    return None


def _first_url(value: Any) -> Optional[str]:
    if isinstance(value, str) and value.startswith(("http://", "https://")):
        return value
    if isinstance(value, dict):
        for item in value.values():
            found = _first_url(item)
            if found:
                return found
    if isinstance(value, list):
        for item in value:
            found = _first_url(item)
            if found:
                return found
    return None


class HostUploader:
    """Uploads like the migration pipeline does, without putting the key on a
    command line: curl reads the URL and form fields from a 0600 config file."""

    def __init__(self, host: str, key: str, session: Optional[requests.Session] = None):
        if host not in HOSTS:
            raise ValueError(f"Unknown host {host}")
        if not key:
            raise RuntimeError(f"Thiếu key {HOSTS[host]['secret']} trong Colab Secrets.")
        self.host, self.key, self.info = host, register_secret(key), HOSTS[host]
        self.session = session or requests.Session()

    def _get(self, path: str, **params: Any) -> Any:
        last = ""
        for attempt in range(5):
            try:
                response = self.session.get(self.info["api"] + path, params={"key": self.key, **params}, timeout=(20, 90))
                if response.status_code == 429 or response.status_code >= 500:
                    last = f"HTTP {response.status_code}"
                    time.sleep(2 ** attempt + 1)
                    continue
                if not response.ok:
                    raise RuntimeError(f"{self.info['label']} HTTP {response.status_code}: {redact(response.text, 200)}")
                return response.json()
            except (requests.RequestException, ValueError) as error:
                last = redact(error)
                time.sleep(2 ** attempt + 1)
        raise RuntimeError(f"{self.info['label']} API lỗi: {last}")

    def _curl(self, url: str, fields: Dict[str, str], path: Path) -> Any:
        if not shutil.which("curl"):
            raise RuntimeError("curl is not installed")

        def quote(value: str) -> str:
            return '"' + value.replace("\\", "\\\\").replace('"', '\\"') + '"'

        config = tempfile.NamedTemporaryFile("w", delete=False, prefix="gv-curl-", suffix=".cfg")
        try:
            os.chmod(config.name, 0o600)
            config.write(f"url = {quote(url)}\n")
            for name, value in fields.items():
                config.write(f"form = {quote(f'{name}={value}')}\n")
            config.write(f"form = {quote(f'file=@{path};filename={path.name};type=video/mp4')}\n")
            config.close()
            result = subprocess.run(
                ["curl", "--silent", "--show-error", "--fail-with-body", "--connect-timeout", "60",
                 "--max-time", "21600", "--retry", "2", "--retry-delay", "10", "--retry-all-errors", "-K", config.name],
                capture_output=True, text=True,
            )
        finally:
            os.unlink(config.name)
        if result.returncode != 0:
            raise RuntimeError(f"{self.info['label']} upload lỗi: {redact(result.stderr or result.stdout, 300)}")
        try:
            return json.loads(result.stdout)
        except ValueError:
            raise RuntimeError(f"{self.info['label']} upload trả về không phải JSON: {redact(result.stdout, 200)}") from None

    def upload(self, path: Path, title: str, tags: Sequence[str] = ()) -> str:
        """Uploads the file and returns the host's file code."""
        server = _first_url(self._get("/upload/server"))
        if not server:
            raise RuntimeError(f"{self.info['label']} không trả về upload server.")
        if self.host == "dood":
            url = server if "?" in server else f"{server}?{self.key}"
            payload = self._curl(url, {"api_key": self.key}, path)
        elif self.host == "voe":
            payload = self._curl(f"{server}{'&' if '?' in server else '?'}key={self.key}", {}, path)
        else:
            payload = self._curl(server, {"key": self.key, "file_title": title, "tags": ",".join(tags),
                                          "file_public": "1", "file_adult": "1"}, path)
        code = _find_value(payload, ("filecode", "file_code", "fileCode", "code"))
        if not isinstance(code, str) or not code:
            raise RuntimeError(f"{self.info['label']} không trả về mã file: {redact(payload, 200)}")
        if title and self.host in ("dood", "voe"):
            try:
                self._get("/file/rename", file_code=code, title=title)
            except Exception:
                pass
        return code


# --- ledger -------------------------------------------------------------------------

LEDGER_FIELDS = ["time", "link", "item_key", "sha256", "title", "video_id", "url", "x", "status", "error"]


class Ledger:
    """A CSV of every item handled, so a link or file is never posted twice
    (kept on Google Drive so it survives Colab sessions)."""

    def __init__(self, path: str):
        self.path = path
        self.rows: List[Dict[str, str]] = []
        if os.path.exists(path):
            with open(path, newline="", encoding="utf-8") as handle:
                self.rows = list(csv.DictReader(handle))

    def done_keys(self) -> set:
        return {row["item_key"] for row in self.rows if row.get("status") == "done"}

    def done_hashes(self) -> set:
        return {row["sha256"] for row in self.rows if row.get("status") == "done" and row.get("sha256")}

    def add(self, **row: str) -> None:
        record = {name: str(row.get(name, "") or "") for name in LEDGER_FIELDS}
        record["time"] = record["time"] or dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        self.rows.append(record)
        os.makedirs(os.path.dirname(self.path) or ".", exist_ok=True)
        new = not os.path.exists(self.path)
        with open(self.path, "a", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=LEDGER_FIELDS)
            if new:
                writer.writeheader()
            writer.writerow(record)

In [ ]:
#@title 1. Cài đặt (chạy một lần mỗi phiên)
!apt-get -qq install -y librsvg2-bin > /dev/null
!pip -q install gdown requests pillow
import importlib, sys
sys.path.insert(0, "/content")
for name in ("colab_helpers", "gayvideo_publish"):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
print("OK: ffmpeg", "có" if __import__("shutil").which("ffmpeg") else "THIẾU")

In [ ]:
#@title 2. Cấu hình
API_URL = "https://gayvideo.fun"  #@param {type:"string"}
#@markdown Lên web ngay (bỏ chọn = bản nháp để duyệt trong /admin/review; bài X chờ tới khi duyệt).
PUBLISH = True  #@param {type:"boolean"}
#@markdown Đăng thêm lên X @Gayvideofun qua Buffer (clip ngắn + link). Độ dài clip theo `X_CLIP_SECONDS` trên server.
POST_TO_X = True  #@param {type:"boolean"}
#@markdown Caption X mặc định (tiếng Anh) khi dòng không có caption riêng; `{title}` = tiêu đề bỏ dấu.
X_CAPTION_TEMPLATE = "New video: {title} 🔥"  #@param {type:"string"}
#@markdown Clip X bắt đầu từ giây thứ (khi dòng không ghi riêng).
CLIP_START = 0  #@param {type:"number"}
#@markdown Gắn watermark GayVideo.fun trước khi upload (giống luồng hiện tại).
WATERMARK = True  #@param {type:"boolean"}
#@markdown Upload thêm lên các host phụ (cần key trong Secrets).
UPLOAD_MIRRORS = False  #@param {type:"boolean"}
#@markdown Lấy thumbnail ở giây thứ (-1 = tự chọn, khoảng 20% thời lượng).
THUMB_AT = -1  #@param {type:"number"}
#@markdown Sổ ghi các video đã đăng (để không đăng trùng). Nên để trên Drive.
MOUNT_DRIVE = True  #@param {type:"boolean"}
LEDGER_PATH = "/content/drive/MyDrive/gayvideo/publish_log.csv"  #@param {type:"string"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
elif LEDGER_PATH.startswith("/content/drive"):
    LEDGER_PATH = "/content/publish_log.csv"
    print("Cảnh báo: không mount Drive, sổ ghi chỉ tồn tại trong phiên này:", LEDGER_PATH)

In [ ]:
#@title 3. Danh sách video
#@markdown Mỗi dòng: `link | tiêu đề | tag1, tag2 | category | caption X (tiếng Anh) | giây bắt đầu clip`. Chỉ link là bắt buộc. Album → mỗi video một bài, tiêu đề tự thêm "– P1, P2…". Dòng bắt đầu bằng `#` bị bỏ qua.
LINKS = """
# https://photos.app.goo.gl/XXXXXXXX | Bevis Trần Đức Vương – My love story | Bevis, Couple | couple | Bevis & his love story 🔥 | 30
"""
#@markdown Hoặc đọc từ CSV (cột: `link,title,tags,category,x_caption,clip_start`); để trống nếu dùng ô trên.
CSV_PATH = ""  #@param {type:"string"}

from colab_helpers import parse_csv, parse_lines
JOBS = parse_csv(CSV_PATH) if CSV_PATH.strip() else parse_lines(LINKS)
print(f"{len(JOBS)} link")
for job in JOBS:
    print(" -", job.link, "|", job.title or "(tiêu đề tự động)")

### 4. Xác nhận (bắt buộc)
Trước khi đăng, bạn phải xác nhận **cả hai** điều sau cho **mọi** video trong danh sách:
1. Bạn có quyền đăng các video này (bạn là chủ sở hữu, hoặc được chủ sở hữu cho phép).
2. Mọi người xuất hiện trong video đều **từ 18 tuổi trở lên** và đã **đồng ý** quay và đăng video.

Không đánh dấu ô dưới đây thì ô Đăng sẽ dừng ngay. Thời điểm xác nhận được ghi vào sổ cùng với từng video.

In [ ]:
#@title Tôi xác nhận có quyền đăng các video này; mọi người trong video đều trên 18 tuổi và đồng ý.
XAC_NHAN = False  #@param {type:"boolean"}
import datetime
XAC_NHAN_LUC = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ") if XAC_NHAN else ""
print("Đã xác nhận lúc", XAC_NHAN_LUC) if XAC_NHAN else print("Chưa xác nhận: ô Đăng sẽ không chạy.")

In [ ]:
#@title 5. Đăng
import shutil, tempfile
from pathlib import Path

import pandas as pd
import requests
from google.colab import userdata
from IPython.display import display

from colab_helpers import (HostUploader, Ledger, SourceError, ascii_text, make_thumbnail, probe,
                           redact, register_secret, sha256_file, source_items, watermark)
from gayvideo_publish import GayVideoClient, PublishError

if not globals().get("XAC_NHAN"):
    raise SystemExit("Dừng: hãy đánh dấu ô xác nhận ở bước 4 rồi chạy lại ô đó.")


def secret(name):
    try:
        return register_secret(userdata.get(name))
    except Exception:
        return None


client = GayVideoClient(token=secret("GAYVIDEO_API_TOKEN"), base_url=API_URL, timeout=900)
meta = client.meta()
primary = next(host["id"] for host in meta["hosts"] if host["isPrimary"])
targets = [primary] + ([h["id"] for h in meta["hosts"] if not h["isPrimary"]] if UPLOAD_MIRRORS else [])
uploaders = {}
for host in targets:
    key = secret({"dood": "HOST_DOOD_API_KEY", "voe": "HOST_VOE_API_KEY",
                  "earnvids": "HOST_EARNVIDS_API_KEY"}.get(host, ""))
    if key:
        uploaders[host] = HostUploader(host, key)
    elif host == primary:
        raise SystemExit(f"Thiếu key của host chính ({host}) trong Colab Secrets.")
    else:
        print(f"Bỏ qua host phụ {host}: chưa có key trong Secrets.")
categories = {c["slug"] for c in meta["categories"]}
if POST_TO_X and not meta.get("x", {}).get("enabled"):
    print("Lưu ý: server chưa cấu hình Buffer; bài X sẽ chờ trong hàng đợi.")
print(f"Host chính: {primary}; upload lên: {', '.join(uploaders)}")

logo = None
if WATERMARK:
    logo = Path("/content/gv-logo.svg")
    logo.write_bytes(requests.get(API_URL.rstrip("/") + "/brand/logo-mark.svg", timeout=60).content)

ledger = Ledger(LEDGER_PATH)
results = []


def note(job, title, status, detail="", url="", x=""):
    results.append({"link": job.link, "tiêu đề": title, "trạng thái": status,
                    "link web": url, "X": x, "ghi chú": redact(detail, 300)})


for job in JOBS:
    try:
        album_title, items = source_items(job.link)
    except Exception as error:  # one bad link never stops the batch
        note(job, job.title, "lỗi", error)
        ledger.add(link=job.link, status="error", error=redact(error))
        continue
    if not items:
        note(job, job.title, "lỗi", "Không có video nào trong link.")
        continue

    for index, item in enumerate(items):
        base_title = job.title or album_title or f"Video {datetime.date.today():%d-%m-%Y}"
        title = f"{base_title} – P{index + 1}" if len(items) > 1 else base_title
        if item.key in ledger.done_keys():
            note(job, title, "bỏ qua", "Đã đăng trước đó (theo link).")
            continue
        work = Path(tempfile.mkdtemp(prefix="gv-job-"))
        codes, digest = {}, ""
        try:
            print(f"▶ {title}: tải video…")
            source = item.download(work / "source.mp4")
            digest = sha256_file(source)
            if digest in ledger.done_hashes():
                note(job, title, "bỏ qua", "Đã đăng trước đó (cùng file).")
                ledger.add(link=job.link, item_key=item.key, sha256=digest, title=title, status="duplicate")
                continue
            info = probe(source)
            upload_file = watermark(source, work / "upload.mp4", logo, info["width"]) if WATERMARK else source

            warnings = []
            for host, uploader in uploaders.items():
                print(f"  upload {host}…")
                try:
                    codes[host] = uploader.upload(upload_file, title, job.tags)
                except Exception as error:
                    if host == primary:
                        raise
                    warnings.append(f"{host}: {redact(error, 150)}")

            thumb = make_thumbnail(source, work / "thumb.jpg", info["duration"], None if THUMB_AT < 0 else THUMB_AT)
            category = job.category if job.category in categories else None
            if job.category and not category:
                warnings.append(f"category '{job.category}' không tồn tại, đã bỏ")
            caption = job.x_caption or X_CAPTION_TEMPLATE.format(title=ascii_text(title))
            print("  publish…")
            result = client.publish(
                title, codes, tags=job.tags or None, category=category, thumbnail=str(thumb), publish=PUBLISH,
                x_caption=caption if POST_TO_X else None, x_video=str(upload_file) if POST_TO_X else None,
                x_clip_start=job.clip_start if job.clip_start is not None else CLIP_START,
            )
            video = result["video"]
            url = API_URL.rstrip("/") + video["url"]
            x_status = (result.get("x") or {}).get("status", "")
            ledger.add(link=job.link, item_key=item.key, sha256=digest, title=title, video_id=video["id"],
                       url=url, x=x_status, status="done", error=f"xác nhận {XAC_NHAN_LUC}")
            note(job, title, "đã đăng" if video["published"] else "bản nháp", "; ".join(warnings + result["warnings"]),
                 url, x_status)
        except Exception as error:
            detail = f"{error.code}: {error}" if isinstance(error, PublishError) else str(error)
            if codes:
                detail += f" (đã upload: {codes}; có thể đăng lại thủ công bằng mã file này)"
            note(job, title, "lỗi", detail)
            ledger.add(link=job.link, item_key=item.key, sha256=digest, title=title, status="error", error=redact(detail))
        finally:
            shutil.rmtree(work, ignore_errors=True)

pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(results))
print("Sổ ghi:", LEDGER_PATH)

### Gặp lỗi?
- **"Google Photos …"**: link chưa chia sẻ công khai, hoặc Google chặn tải tự động. Dùng link Google Drive thay thế (xem đầu notebook).
- **"Thiếu key …"**: thêm secret tương ứng trong 🔑 Secrets và bật *Notebook access*.
- **`401 unauthorized`**: token sai hoặc đã bị thu hồi. Tạo token mới trên VPS: `npm run api-token -- create <email> colab`.
- **Lỗi sau khi đã upload host:** bảng kết quả có ghi mã file đã upload. Chạy lại notebook sẽ upload lại; muốn tránh thì đăng thủ công bằng mã file đó (xem `docs/API.md`).
- **Xem bài X:** `GET /api/v1/videos/<id>` (trường `social`), hoặc trên VPS chạy `npm run social:status`.